# Aggregate Classification Sweep Results

Load classification sweep results for multiple campaigns, select one top percentage, and average the results across campaigns.

In [ ]:
from pathlib import Path

import pandas as pd

## Parameters

In [ ]:
Campaigns = ["Iran_1", "UAE", "Venezuela", "China_1", "Cuba", "Russia_1"]
top_percentage = 100

In [ ]:
def find_project_root(start_path):
    """Find the project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the project root containing AGENTS.md.")


def get_results_path(project_root, campaign):
    """Build the classification sweep results path for a campaign.

    Args:
        project_root: Root directory of the project.
        campaign: Campaign directory name.

    Returns:
        Path to the campaign's classification sweep CSV file.
    """
    return (
        project_root
        / "results"
        / "Labeled_Datasets"
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / "classification_sweep"
        / "classification_sweep_results.csv"
    )

In [ ]:
if not Campaigns:
    raise ValueError("Campaigns must contain at least one campaign name.")

project_root = find_project_root(Path.cwd())
campaign_results = {}

for campaign in Campaigns:
    results_path = get_results_path(project_root, campaign)
    campaign_df = pd.read_csv(results_path)
    campaign_df = campaign_df.loc[
        campaign_df["top_percentage"].eq(top_percentage)
    ]
    campaign_df["indicators_configuration"] = campaign_df[
        "indicators_configuration"
    ].str.replace(
        " (Unweighted Clustering, No Square Clustering)",
        "",
        regex=False,
    )
    campaign_results[campaign] = campaign_df

configuration_columns = [
    "topic_col",
    "account_relations_configuration",
    "indicators_configuration",
]
configuration_presence_records = []
common_configuration_values = {}

for configuration_column in configuration_columns:
    values_by_campaign = {
        campaign: set(campaign_df[configuration_column].dropna().unique())
        for campaign, campaign_df in campaign_results.items()
    }
    common_configuration_values[configuration_column] = set.intersection(
        *values_by_campaign.values()
    )
    all_configuration_values = set.union(*values_by_campaign.values())

    for configuration_value in sorted(all_configuration_values):
        present_in = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value in values
        ]
        missing_from = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value not in values
        ]
        configuration_presence_records.append(
            {
                "configuration": configuration_column,
                "value": configuration_value,
                "present_in": ", ".join(present_in),
                "missing_from": ", ".join(missing_from) or "None",
                "present_in_all_datasets": not missing_from,
            }
        )

configuration_presence_df = pd.DataFrame(configuration_presence_records)
print("Configuration availability by campaign:")
display(configuration_presence_df)

requested_summary_metric_columns = {
    "all_authors_auc": "ROC_AUC",
    "all_authors_auc_pr": "PR_AUC",
}
metric_presence_records = []
summary_metric_columns = {}
for metric_column, metric_name in requested_summary_metric_columns.items():
    present_in = [
        campaign
        for campaign, campaign_df in campaign_results.items()
        if metric_column in campaign_df
        and campaign_df[metric_column].notna().any()
    ]
    missing_from = [
        campaign for campaign in Campaigns if campaign not in present_in
    ]
    metric_presence_records.append(
        {
            "metric": metric_name,
            "source_column": metric_column,
            "present_in": ", ".join(present_in),
            "missing_from": ", ".join(missing_from) or "None",
            "present_in_all_datasets": not missing_from,
        }
    )
    if not missing_from:
        summary_metric_columns[metric_column] = metric_name

metric_presence_df = pd.DataFrame(metric_presence_records)
print("Metric availability by campaign:")
display(metric_presence_df)

all_results_df = pd.concat(campaign_results.values(), ignore_index=True)
for configuration_column, common_values in common_configuration_values.items():
    all_results_df = all_results_df.loc[
        all_results_df[configuration_column].isin(common_values)
    ]
all_results_df = all_results_df.reset_index(drop=True)
group_columns = all_results_df.select_dtypes(exclude="number").columns.tolist()
aggregated_df = (
    all_results_df.groupby(group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(group_columns)
    .reset_index(drop=True)
)

display(aggregated_df)

classification_group_columns = configuration_columns
classification_metric_columns = [
    column
    for column in all_results_df.columns
    if column.startswith(("graph_authors_", "all_authors_"))
    and all(
        column in campaign_df and campaign_df[column].notna().any()
        for campaign_df in campaign_results.values()
    )
]
classification_metrics_df = (
    all_results_df[
        classification_group_columns + classification_metric_columns
    ]
    .groupby(classification_group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(classification_group_columns)
    .reset_index(drop=True)
)

display(classification_metrics_df)

topic_col_aggregation_df = (
    all_results_df.groupby("topic_col", as_index=False, dropna=False)[
        list(summary_metric_columns)
    ]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("topic_col")
    .reset_index(drop=True)
)
display(topic_col_aggregation_df)

account_relations_aggregation_df = (
    all_results_df.groupby(
        "account_relations_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns)]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("account_relations_configuration")
    .reset_index(drop=True)
)
display(account_relations_aggregation_df)

indicators_aggregation_df = (
    all_results_df.groupby(
        "indicators_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns) + ["indicators_runtime_seconds"]]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("indicators_configuration")
    .reset_index(drop=True)
)
display(indicators_aggregation_df)

In [ ]:
campaigns_name = "_".join(Campaigns).lower().replace(" ", "_")
percentage_name = f"{top_percentage:g}"
output_folder_name = f"top_percentage_{percentage_name}_{campaigns_name}"
output_dir = (
    project_root
    / "results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated"
    / output_folder_name
)
output_dir.mkdir(parents=True, exist_ok=True)

output_stem = "aggregated_classification_sweep_results"
csv_path = output_dir / f"{output_stem}.csv"
latex_path = output_dir / f"{output_stem}.tex"
metrics_output_stem = "aggregated_classification_metrics"
metrics_csv_path = output_dir / f"{metrics_output_stem}.csv"
metrics_latex_path = output_dir / f"{metrics_output_stem}.tex"

aggregated_df.to_csv(csv_path, index=False)
aggregated_df.to_latex(latex_path, index=False, float_format="%.3f")
classification_metrics_df.to_csv(metrics_csv_path, index=False)
classification_metrics_df.to_latex(
    metrics_latex_path, index=False, float_format="%.3f"
)

summary_aggregations = {
    "topic_col": topic_col_aggregation_df,
    "account_relations_configuration": account_relations_aggregation_df,
    "indicators_configuration": indicators_aggregation_df,
}
for aggregation_name, summary_df in summary_aggregations.items():
    summary_output_stem = f"{output_stem}_by_{aggregation_name}"
    summary_csv_path = output_dir / f"{summary_output_stem}.csv"
    summary_latex_path = output_dir / f"{summary_output_stem}.tex"
    summary_df.to_csv(summary_csv_path, index=False)
    summary_df.to_latex(
        summary_latex_path, index=False, float_format="%.3f"
    )
    print(f"Saved summary CSV: {summary_csv_path}")
    print(f"Saved summary LaTeX: {summary_latex_path}")

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX: {latex_path}")
print(f"Saved classification metrics CSV: {metrics_csv_path}")
print(f"Saved classification metrics LaTeX: {metrics_latex_path}")